# 04 — Comparing biological-age posterior diagnostics

This notebook runs the four diagnostic configurations through one common training and evaluation loop. It compares **point-estimate accuracy**, **posterior calibration**, **informativeness**, and **posterior-predictive adequacy**. The network design and training budget are held fixed; the intended changes are simulator complexity, noise, and data source.

The default mode uses the configured budgets and reuses an existing artifact when available. Set `BIOAGE_DIAGNOSTIC_RUN_MODE=smoke` for a small end-to-end check whose outputs go to `/tmp` rather than the experiment result folders.

In [ ]:
from __future__ import annotations

import copy
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in [HERE, *HERE.parents]
    if (candidate / 'biological_age_sbi' / 'experiment').exists()
)
EXPERIMENT_ROOT = PROJECT_ROOT / 'biological_age_sbi' / 'experiment'
SRC_ROOT = EXPERIMENT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bioage_sbi.variant_experiment import (
    load_variant_config,
    load_variant_results,
    run_diagnostic_variant,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Select the variants and execution mode

`configured` runs all four 32,768-simulation experiments unless their artifacts already exist. `smoke` uses tiny budgets and redirected output paths. `BIOAGE_DIAGNOSTIC_MAX_VARIANTS` can temporarily limit the list during development.

In [ ]:
CONFIG_NAMES = [
    'bioage_diag_01_simple_mendeley.json',
    'bioage_diag_02_advanced_mendeley.json',
    'bioage_diag_03_advanced_reduced_noise_mendeley.json',
    'bioage_diag_04_advanced_reduced_noise_combined.json',
]
RUN_MODE = os.environ.get('BIOAGE_DIAGNOSTIC_RUN_MODE', 'configured').lower()
REUSE_EXISTING = os.environ.get('BIOAGE_DIAGNOSTIC_REUSE', '1') == '1'
MAX_VARIANTS = int(os.environ.get('BIOAGE_DIAGNOSTIC_MAX_VARIANTS', len(CONFIG_NAMES)))
if RUN_MODE not in {'configured', 'smoke'}:
    raise ValueError("BIOAGE_DIAGNOSTIC_RUN_MODE must be 'configured' or 'smoke'.")

configs = [
    load_variant_config(EXPERIMENT_ROOT / 'configs' / name)
    for name in CONFIG_NAMES[:MAX_VARIANTS]
]
LABELS = {
    '01_simple_mendeley': '1 Simple · Mendeley',
    '02_advanced_mendeley': '2 Advanced · Mendeley',
    '03_advanced_reduced_noise_mendeley': '3 Advanced + low noise · Mendeley',
    '04_advanced_reduced_noise_combined': '4 Advanced + low noise · combined',
}
COLORS = dict(zip(LABELS, plt.get_cmap('tab10').colors))

design_rows = []
for config in configs:
    sim = config['simulator_config']
    bio = sim['bioindicator_model']
    design_rows.append({
        'variant': config['variant'],
        'data_source': sim['data_source']['name'],
        'model': sim['model_name'],
        'copula': bio['copula_enabled'],
        'latent_factors': bio['latent_factors_enabled'],
        'residual_noise_scale': bio['continuous_residual_noise_scale'],
        'latent_std_scale': bio['latent_std_scale'],
        'observation_noise_scale': sim['observation_model']['continuous_noise_scale'],
        'training_simulations': config['training_config']['num_simulations'],
    })
design_table = pd.DataFrame(design_rows)
print(f'Run mode: {RUN_MODE}; reuse existing: {REUSE_EXISTING}')
display(design_table)

## 2. Train or reuse every variant

Each run gets independent seeded simulator streams for training, validation, calibration, and posterior-predictive simulation. The evaluation set is synthetic, so every row has its own known ground-truth biological age.

In [ ]:
SMOKE_OVERRIDES = {
    'num_simulations': 128,
    'num_validation_simulations': 64,
    'epochs': 1,
    'batch_size': 32,
    'num_calibration_cases': 64,
    'num_posterior_predictive_samples': 12,
    'num_posterior_predictive_cases': 8,
}

def prepared_config(config):
    run_config = copy.deepcopy(config)
    if RUN_MODE == 'smoke':
        output_root = Path('/tmp/bioage_diagnostic_smoke') / run_config['variant']
        run_config['training_config']['checkpoint_path'] = str(output_root / 'checkpoint.keras')
        run_config['evaluation_config']['result_dir'] = str(output_root)
        run_config['evaluation_config']['artifact_path'] = str(output_root / 'evaluation_arrays.npz')
    return run_config

results = {}
for original_config in configs:
    config = prepared_config(original_config)
    variant = config['variant']
    artifact_path = Path(config['evaluation_config']['artifact_path'])
    if not artifact_path.is_absolute():
        artifact_path = PROJECT_ROOT / artifact_path
    if REUSE_EXISTING and artifact_path.exists():
        print(f'[{variant}] loading {artifact_path}')
        result = load_variant_results(config, PROJECT_ROOT)
    else:
        print(f'[{variant}] training and evaluating')
        overrides = SMOKE_OVERRIDES if RUN_MODE == 'smoke' else None
        result = run_diagnostic_variant(config, PROJECT_ROOT, budget_overrides=overrides)
    results[variant] = result

COMPARISON_DIR = (
    Path('/tmp/bioage_diagnostic_smoke/comparison')
    if RUN_MODE == 'smoke'
    else EXPERIMENT_ROOT / 'results' / 'posterior_diagnostics' / 'comparison'
)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
print(f'Comparison outputs: {COMPARISON_DIR}')

## 3. Compact comparison table

The table keeps the four diagnostic angles separate. Lower is better for RMSE, calibration error, crossing rate, and predictive scores; coverage should be close to its nominal level; larger positive contraction means narrower posterior intervals than the prior.

In [ ]:
summary_rows = []
for variant, result in results.items():
    point = result['point_metrics']
    interval80 = result['intervals'].loc[
        np.isclose(result['intervals']['interval_level'], 0.8)
    ].iloc[0]
    continuous = result['continuous_ppc']
    binary = result['binary_ppc']
    summary_rows.append({
        'variant': variant,
        'correlation_r': point['correlation_r'],
        'mae': point['mae'],
        'rmse': point['rmse'],
        'normalized_rmse': point['prior_std_normalized_rmse'],
        'bias': point['bias'],
        'mean_abs_calibration_error': result['calibration']['absolute_calibration_error'].mean(),
        'quantile_crossing_rate': result['quantile_crossing_rate'],
        'coverage_80': interval80['empirical_coverage'],
        'mean_width_80': interval80['mean_width'],
        'contraction_80': interval80['mean_quantile_contraction'],
        'continuous_ppc_normalized_rmse': (
            continuous['observed_std_normalized_rmse'].mean() if continuous is not None else np.nan
        ),
        'continuous_ppc_coverage_80': (
            continuous['predictive_coverage'].mean() if continuous is not None else np.nan
        ),
        'binary_ppc_brier': binary['brier_score'].mean() if binary is not None else np.nan,
    })
summary = pd.DataFrame(summary_rows).set_index('variant')
display(summary.round(3))
summary.to_csv(COMPARISON_DIR / 'variant_diagnostic_summary.csv')

## 4. Training behavior and point-estimate recovery

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for variant, result in results.items():
    history = result['history']
    if 'loss' in history:
        ax.plot(np.arange(1, len(history) + 1), history['loss'], label=LABELS[variant], color=COLORS[variant])
    if 'val_loss' in history:
        ax.plot(np.arange(1, len(history) + 1), history['val_loss'], linestyle='--', color=COLORS[variant], alpha=0.8)
ax.set(xlabel='Epoch', ylabel='Scoring-rule loss', title='Training loss (solid) and validation loss (dashed)')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(COMPARISON_DIR / 'training_loss_comparison.png', dpi=160)
plt.show()

n_variants = len(results)
fig, axes = plt.subplots(1, n_variants, figsize=(4 * n_variants, 4), squeeze=False, sharex=True, sharey=True)
all_values = np.concatenate([np.r_[r['truth'], r['point_estimate']] for r in results.values()])
limits = [float(all_values.min()), float(all_values.max())]
for ax, (variant, result) in zip(axes.flat, results.items()):
    ax.scatter(result['truth'], result['point_estimate'], s=10, alpha=0.25, color=COLORS[variant])
    ax.plot(limits, limits, color='black', linewidth=1)
    metrics = result['point_metrics']
    ax.set_title(f"{LABELS[variant]}\nr={metrics['correlation_r']:.2f}, RMSE={metrics['rmse']:.2f}", fontsize=9)
    ax.set(xlabel='True biological age', ylabel='Estimated posterior mean')
fig.suptitle('Held-out synthetic truth recovery', y=1.03)
fig.tight_layout()
fig.savefig(COMPARISON_DIR / 'point_recovery_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
plot_summary = summary.reset_index()
x = np.arange(len(plot_summary))
short_labels = [str(i + 1) for i in range(len(plot_summary))]
for ax, column, title in zip(
    axes,
    ['correlation_r', 'rmse', 'bias'],
    ['Correlation (higher is better)', 'RMSE (lower is better)', 'Bias (zero is ideal)'],
):
    ax.bar(x, plot_summary[column], color=[COLORS[v] for v in plot_summary['variant']])
    ax.set_xticks(x, short_labels)
    ax.set_title(title, fontsize=10)
axes[2].axhline(0, color='black', linewidth=1)
fig.suptitle('Point-estimate accuracy by variant number')
fig.tight_layout()
fig.savefig(COMPARISON_DIR / 'point_metric_comparison.png', dpi=160)
plt.show()

## 5. Posterior calibration

For each nominal quantile $q$, the curve shows $P(\theta_{true} \leq \hat Q_q(x)) - q$ across independently simulated test datasets. Zero is ideal. The grey region is the pointwise binomial sampling band, not posterior uncertainty.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
first = next(iter(results.values()))['calibration']
q = first['quantile_level'].to_numpy()
ax.fill_between(
    q,
    first['pointwise_lower'].to_numpy() - q,
    first['pointwise_upper'].to_numpy() - q,
    color='0.85',
    label=f"{configs[0]['evaluation_config']['pointwise_confidence']:.0%} pointwise band",
)
for variant, result in results.items():
    calibration = result['calibration']
    ax.plot(
        calibration['quantile_level'], calibration['calibration_error'],
        marker='o', label=LABELS[variant], color=COLORS[variant],
    )
ax.axhline(0, color='black', linewidth=1)
ax.set(xlabel='Nominal posterior quantile', ylabel='Empirical frequency − nominal quantile', title='Marginal quantile calibration')
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(COMPARISON_DIR / 'quantile_calibration_comparison.png', dpi=160)
plt.show()

coverage_rows = []
for variant, result in results.items():
    frame = result['intervals'].copy()
    frame['variant'] = variant
    coverage_rows.append(frame)
coverage = pd.concat(coverage_rows, ignore_index=True)
fig, ax = plt.subplots(figsize=(7, 4.5))
for variant, group in coverage.groupby('variant', sort=False):
    ax.plot(group['interval_level'], group['empirical_coverage'], marker='o', label=LABELS[variant], color=COLORS[variant])
ax.plot([0.45, 0.95], [0.45, 0.95], color='black', linestyle='--', linewidth=1, label='Ideal')
ax.set(xlabel='Nominal central interval', ylabel='Empirical coverage', title='Credible-interval coverage')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(COMPARISON_DIR / 'interval_coverage_comparison.png', dpi=160)
plt.show()

## 6. Informativeness

Calibration and narrowness must be read together. `mean_quantile_contraction = 1 - posterior width / prior width`; values near one indicate strong contraction, zero means no narrowing relative to the prior, and negative values mean expansion.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for variant, group in coverage.groupby('variant', sort=False):
    axes[0].plot(group['interval_level'], group['mean_width'], marker='o', label=LABELS[variant], color=COLORS[variant])
    axes[1].plot(group['interval_level'], group['mean_quantile_contraction'], marker='o', label=LABELS[variant], color=COLORS[variant])
axes[0].set(xlabel='Central interval level', ylabel='Mean interval width', title='Posterior interval width')
axes[1].set(xlabel='Central interval level', ylabel='Contraction relative to prior', title='Posterior contraction')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(COMPARISON_DIR / 'informativeness_comparison.png', dpi=160)
plt.show()

## 7. Posterior-predictive adequacy

These checks ask whether ages drawn from the estimated conditional distribution generate plausible observations under that variant's simulator. Because variants 1–3 and 4 use different feature sets/data sources, compare their aggregate scores cautiously and inspect the saved feature-level CSV files when diagnosing a specific mismatch.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
ppc_columns = [
    ('continuous_ppc_normalized_rmse', 'Continuous normalized RMSE', None),
    ('continuous_ppc_coverage_80', 'Continuous 80% coverage', 0.8),
    ('binary_ppc_brier', 'Binary Brier score', None),
]
for ax, (column, title, target) in zip(axes, ppc_columns):
    ax.bar(x, plot_summary[column], color=[COLORS[v] for v in plot_summary['variant']])
    ax.set_xticks(x, short_labels)
    ax.set_title(title, fontsize=10)
    if target is not None:
        ax.axhline(target, color='black', linestyle='--', linewidth=1)
fig.suptitle('Aggregate posterior-predictive diagnostics by variant number')
fig.tight_layout()
fig.savefig(COMPARISON_DIR / 'posterior_predictive_comparison.png', dpi=160)
plt.show()

## Interpretation checklist

1. Check that training and validation loss converge before interpreting posterior failures.
2. Read recovery and calibration together: accurate means can coexist with miscalibrated uncertainty.
3. Read calibration and informativeness together: prior-like intervals can be calibrated but unhelpful.
4. Treat quantile crossings as a network-output failure. Raw quantiles are used for calibration; monotone rearrangement is used only to make the approximate PPC sampling bridge defined.
5. A better synthetic diagnostic does not establish real-data validity. Simulator–real mismatch remains a separate check.
6. Variant 4 changes both the data source/prior fit and the available feature set, so it is a practical baseline comparison rather than a clean one-factor ablation.